In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Employee Salary Prediction – Model Development

## Project Overview

This project is part of a broader analysis titled **Employee Salary, Satisfaction & Attrition Analysis**. The primary objective of this notebook is to build and evaluate machine learning models to accurately predict the **annual salary of employees** based on various professional and demographic features.

The problem is approached as a **supervised regression task**, where the model learns patterns from historical employee data and estimates salary for new or unseen profiles.

## Dataset

The dataset used in this project contains detailed information about employees, including:

* Experience (total and in-field)
* Education level
* Job role and primary tech field
* Skills, certifications, and languages
* Employment type and company size

**Dataset Source:**
https://www.kaggle.com/datasets/mabubakrsiddiq/salary-and-skills-ds-what-factors-increases-salary

---

## Exploratory Data Analysis (EDA)

Before model building, a comprehensive EDA was performed to:

* Clean and preprocess the data
* Handle missing values and categorical inconsistencies
* Engineer new features (e.g., number of certifications, languages)
* Analyze relationships between features and salary
* Extract initial business insights

**EDA Notebook:**
(https://www.kaggle.com/code/salonipandagale/eda-for-annual-salary-prediction)

## Objective of This Notebook

In this notebook, we will:

1. Load the cleaned dataset prepared during EDA
2. Perform necessary preprocessing (encoding, scaling)
3. Train multiple regression models (Linear Regression, Random Forest, XGBoost, etc.)
4. Evaluate model performance using appropriate metrics
5. Optimize models using hyperparameter tuning
6. Identify key factors influencing salary through feature importance

## Project Repository

Complete project:
https://github.com/salonipandagale/Employee-Salary-Satisfaction-Attrition-Analysis

## Expected Outcome

By the end of this notebook, we aim to develop a robust salary prediction model and gain deeper insights into the factors that significantly influence employee compensation.

# Load the Cleaned Dataset

In [ ]:
df = pd.read_csv('/kaggle/input/datasets/salonipandagale/cleaned-data-after-eda/cleaned_dataset (2).csv')
df.sample(5)

# Handling high dimensionality

In [ ]:
df['certifications'].value_counts()


1. The certifications, languages_spoken and skills column had high cardinality with many combinations, so direct encoding was not suitable.
2. From languages_spoken extracted meaningful features like the number of languages and a binary indicator for English proficiency. After feature extraction, I dropped the original column to reduce noise and dimensionality.
3. skills column - we have meaningful features representing skills, so we drop "skills" column as well.
4. The certifications column- extracted meaningful features by creating a count of certifications and binary indicators for high-value certifications like AWS, GCP, and TensorFlow.

In [ ]:
#lets create a seperate columns that checks if employee knows English
#as English is a global language
df['knows_english'] = df['languages_spoken'].apply(lambda x: 'English' in str(x))
df['knows_english'] = df['knows_english'].astype(int)

#binary features
df['cert_tensorflow'] = df['certifications'].apply(lambda x: 'TensorFlow' in str(x)).astype(int)
df['cert_aws'] = df['certifications'].apply(lambda x: 'AWS' in str(x)).astype(int)
df['cert_gcp'] = df['certifications'].apply(lambda x: 'Google Cloud' in str(x)).astype(int)
df['cert_pmp'] = df['certifications'].apply(lambda x: 'PMP' in str(x)).astype(int)
df['cert_security'] = df['certifications'].apply(lambda x: 'CEH' in str(x) or 'CISSP' in str(x)).astype(int)

#Drop those three columns
df =df.drop(columns = ['languages_spoken', 'skills', 'certifications'])

#lets check
df.sample()

In [ ]:
#Columns in dataset
df.columns.tolist()

#seperating numerical and categorical columns

num_cols = df.select_dtypes(exclude = 'object').columns.tolist()
cat_cols = df.select_dtypes(include = 'object').columns.tolist()

print("Numerical_columns :",num_cols, "\n")
print("Categorical_columns :", cat_cols)

In [ ]:
df.describe()

1. Annual Salary columns is skewed , so log transformation needed
2. StandardScaler for: age, experience_years_total, experience_years_in_field
3. All cat_cols should under go OneHot Encoding
4. No need to scale/transform - 'num_languages', 'num_certifications', 'total_skills + certificates', 'knows_english', 'cert_tensorflow', 'cert_aws', 'cert_gcp', 'cert_pmp', 'cert_security'

# Install dependencies

In [ ]:
import sklearn
from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Train-test Split

In [ ]:
x = df.drop(columns = 'annual_salary_usd')
y = df['annual_salary_usd']

#log transform y
y = np.log1p(df['annual_salary_usd'])

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size = 0.2, random_state = 42)

# Feature Scaling & Encoding
## Column Transformer

In [ ]:
std_sc_cols = ['age', 'experience_years_total', 'experience_years_in_field']
ohe_cols = ['gender', 'primary_tech_field', 'job_title', 'employment_type', 'work_arrangement', 'education_level', 'primary_skill', 'company_size']
preprocessor = ColumnTransformer(
    transformers =[
        ('num', StandardScaler(), std_sc_cols),
        ('cat',OneHotEncoder(handle_unknown='ignore'), ohe_cols)
    ],
    remainder='passthrough'
)

# Model Pipeline

In [ ]:
#defining mmodel
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for name, model in models.items():
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Train
    pipeline.fit(x_train, y_train)
    
    # Predict (log scale)
    y_pred_log = pipeline.predict(x_test)
    
    # Convert back to original scale
    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)
    
    # Metrics (on original salary scale)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2
    })

# Results of Different Models

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="R2 Score", ascending=False)

results_df

# XGBoost

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb_model)
])

# Train
pipeline.fit(x_train, y_train)

# Predict (log scale)
y_pred_log = pipeline.predict(x_test)

# Convert back
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("XGBoost Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

# Comparison with XGB

In [ ]:
results.append({
    "Model": "XGBoost",
    "MAE": mae,
    "RMSE": rmse,
    "R2 Score": r2
})

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
results_df

# Feature Importance plot

In [ ]:
# Get feature names from preprocessor
ohe_feature_names = preprocessor.named_transformers_['cat'].get_feature_names_out(ohe_cols)

all_feature_names = list(std_sc_cols) + list(ohe_feature_names) + list(
    set(x_train.columns) - set(std_sc_cols) - set(ohe_cols)
)

# Get trained model from pipeline
model = pipeline.named_steps['model']

importances = model.feature_importances_

#create dataframe 
feat_imp_df = pd.DataFrame({
    'Feature': all_feature_names,
    'Importance': importances
})

feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(15)

#create a chart
import matplotlib.pyplot as plt

plt.figure()
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'])
plt.gca().invert_yaxis()
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

# Hyperparameter Optimization of XGB

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
xgb = XGBRegressor(random_state=42, n_jobs=-1)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb)
])

param_dist = {
    'model__n_estimators': [100, 200, 300, 500],
    'model__max_depth': [3, 4, 5, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.6, 0.8, 1.0],
    'model__colsample_bytree': [0.6, 0.8, 1.0],
    'model__gamma': [0, 0.1, 0.3, 0.5],
    'model__reg_alpha': [0, 0.1, 1],
    'model__reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=50,                # increase to 50 for better results
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(x_train, y_train)

best_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

# Evaluate Tuned Model

In [ ]:
# Predict (log scale)
y_pred_log = best_model.predict(x_test)

# Convert back
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test)

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print("\nTuned XGBoost Results:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

# Overall Results

| Model             | R²          |
| ----------------- | ----------- |
| Gradient Boosting | **0.438**   |
| Tuned XGBoost     | **0.453**  |
| Ridge             | 0.398       |
| Linear            | 0.397       |
| Random Forest     | 0.366       |


1. After evaluating multiple models, tree-based ensemble methods significantly outperformed linear models, indicating non-linear relationships in the dataset.
2. Hyperparameter tuning of XGBoost further improved performance, achieving the best results with an R² score of 0.45.
3. However, performance gains plateaued, suggesting that model improvement is now limited by feature quality rather than algorithm choice.
4. Feature importance analysis revealed that factors such as primary technology field, education level, and company size are the most influential predictors of salary.